# 🧠 MFFT-Base — Kaggle Training (Step 2/8)
**Multi-Frequency Fusion Transformer · AI Image Detection Research**

> ⚠️ **This notebook creates `split_indices.json`** — the shared train/val/test split used by ALL other notebooks. Run this BEFORE notebooks 3–8 if you haven't already.

## ⚙️ Setup
1. **Datasets** — Add all 12 datasets listed below (Input → Add Data)
2. **Secret** — Add `HF_TOKEN` secret (Add-ons → Secrets)
3. **Accelerator** — GPU T4 x2
4. **Run All**

### Datasets to Add
| Slug | URL |
|---|---|
| `mihmahmud/stable-diffusion` | stable-diffusion |
| `benjaminkz/places365` | places365 |
| `mihmahmud/open-images-v7-dataset` | open-images-v7 |
| `mihmahmud/ntire2026` | ntire2026 |
| `mihmahmud/midjourney` | midjourney |
| `studyhub991/mfft-real` | mfft-real |
| `studyhub991/genimage-ai` | genimage-ai |
| `greatgamedota/faceforensics` | faceforensics |
| `itamargr/dfdc-faces-of-the-train-sample` | dfdc-faces |
| `mihmahmud/dall-e3` | dall-e3 |
| `pranabr0y/celebdf-v2image-dataset` | celebdf-v2 |
| `awsaf49/artifact-dataset` | artifact-dataset |

**Sequence**: Tiny → **Base** → Large → Baselines → Ablation → PaperEvals → LOGO → Outputs

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 0: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

REPO_URL  = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--rebase"], check=False)

os.chdir(str(CLONE_DIR))
sys.path.insert(0, str(CLONE_DIR / "model"))

subprocess.run([sys.executable, "-m", "pip", "install",
    "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn",
    "-q", "--disable-pip-version-check"], check=False)

print(f"Project root: {CLONE_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Imports & Environment
# ═══════════════════════════════════════════════════════════
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path("/kaggle/working/ai-image-detection-research/model")))
from src.kaggle_utils import KaggleEnv

env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_TEST = not torch.cuda.is_available()
print(f"Device: {device} | SMOKE: {SMOKE_TEST} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Config
# ═══════════════════════════════════════════════════════════
from src.dataset import AIDetectionDataset, ImageTransform, create_split_dataloaders
from src.config import Config
import src
PROJECT_ROOT = Path(src.__file__).resolve().parent.parent.parent

VARIANT     = "base"
NUM_EPOCHS  = 1   if SMOKE_TEST else 20
IMAGE_SIZE  = 224 if SMOKE_TEST else 384
BATCH_SIZE  = 8   if SMOKE_TEST else 64
NUM_WORKERS = 0   if SMOKE_TEST else 4
MAX_SAMPLES = 600 if SMOKE_TEST else None

cfg = Config()
cfg.training.model_variant  = VARIANT
cfg.training.epochs         = NUM_EPOCHS
cfg.training.image_size     = IMAGE_SIZE
cfg.training.batch_size     = BATCH_SIZE
cfg.training.num_workers    = NUM_WORKERS
cfg.training.mixed_precision = torch.cuda.is_available()
cfg.dataset.val_split  = 0.10
cfg.dataset.test_split = 0.10

# ── Manifest ──
_manifest = PROJECT_ROOT / "dataset" / "metadata" / "train_manifest.csv"
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.download_manifest(_manifest)
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.rebuild_manifest_from_kaggle(_manifest)
if _manifest.exists():
    cfg.dataset.metadata_paths = [str(_manifest)]
    print(f"Manifest: {_manifest.stat().st_size/1e6:.1f} MB")

CKPT_DIR = PROJECT_ROOT / "model" / "checkpoints" / f"{VARIANT}_model"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Variant={VARIANT} epochs={NUM_EPOCHS} size={IMAGE_SIZE} batch={BATCH_SIZE}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Data Loaders — Creates shared split_indices.json
# ═══════════════════════════════════════════════════════════
_split_name = "split_indices_smoke.json" if SMOKE_TEST else "split_indices.json"
_split_path = PROJECT_ROOT / "dataset" / "metadata" / _split_name

# NOTE: If split_indices.json does not exist it will be CREATED here.
# All other notebooks (tiny, large, baselines, ablation) download it from HF.
train_loader, val_loader, test_loader = create_split_dataloaders(
    root_dir      = str(PROJECT_ROOT),
    metadata_paths=[str(_manifest)],
    batch_size    = BATCH_SIZE,
    num_workers   = NUM_WORKERS,
    size          = IMAGE_SIZE,
    val_split     = cfg.dataset.val_split,
    test_split    = cfg.dataset.test_split,
    seed          = SEED,
    use_weighted_sampler = True,
    split_index_path     = str(_split_path),
    max_samples          = MAX_SAMPLES,
)
train_dataset = train_loader.dataset
val_dataset   = val_loader.dataset
test_dataset  = test_loader.dataset

class _FullView:
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
full_dataset = _FullView(train_dataset.samples + val_dataset.samples + test_dataset.samples)

print(f"Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Full: {len(full_dataset)} samples")

# Upload split_indices so other notebooks can use it
if _split_path.exists() and not SMOKE_TEST and env.hf_token:
    from huggingface_hub import HfApi
    api = HfApi(token=env.hf_token)
    api.create_repo(env.hf_manifest_repo, repo_type="model", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(_split_path),
        path_in_repo="split_indices.json",
        repo_id=env.hf_manifest_repo,
        repo_type="model",
        commit_message="Upload shared split indices"
    )
    print("split_indices.json uploaded to HF")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Build Model
# ═══════════════════════════════════════════════════════════
from src.model import build_mfft, count_parameters

model = build_mfft(VARIANT).to(device)
print(f"MFFT-{VARIANT.title()} | Parameters: {count_parameters(model):,}")

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Sanity Check
# ═══════════════════════════════════════════════════════════
model.train()
imgs, lbls = next(iter(train_loader))
imgs, lbls = imgs.to(device), lbls.to(device)
logits = model(imgs)
loss   = criterion(logits, lbls)
loss.backward(); optimizer.zero_grad()
print(f"Sanity OK: logits {tuple(logits.shape)}, loss {loss.item():.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6: Full Training Loop — Crash-Safe + HF Resume
# ═══════════════════════════════════════════════════════════
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

AMP    = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

model     = build_mfft(VARIANT).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
warmup    = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
cosine    = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS*len(train_loader), T_mult=2, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

history     = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
start_epoch = 0
best_acc    = 0.0

# ── Resume: local → HF ──
resume_ckpt = env.find_resume_checkpoint(CKPT_DIR)
if resume_ckpt is None:
    resume_ckpt = env.download_latest_checkpoint(CKPT_DIR, f"{VARIANT}_model")
if resume_ckpt is not None:
    state = env.load_checkpoint(resume_ckpt, model, optimizer, scheduler, scaler, device=device)
    if state:
        start_epoch = state["epoch"] + 1
        best_acc    = state["best_acc"]
        history     = state["history"] or history
        print(f"Resuming from epoch {start_epoch}, best={best_acc:.2f}%")
else:
    print("Training from scratch")

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = correct = total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]")
    for images, labels in pbar:
        try:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=AMP):
                logits = model(images); loss = criterion(logits, labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()
            total_loss += loss.item()
            correct += (logits.argmax(-1) == labels).sum().item(); total += labels.size(0)
            pbar.set_postfix({"loss": f"{total_loss/max(1,total//BATCH_SIZE):.4f}",
                              "acc": f"{correct/total*100:.2f}%",
                              "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
        except Exception as e:
            print(f"  Skip bad batch: {e}"); optimizer.zero_grad()

    train_acc = correct / total * 100
    history["train_acc"].append(train_acc)
    history["train_loss"].append(total_loss / len(train_loader))

    model.eval(); val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [val]", leave=False):
            try:
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                val_loss    += criterion(logits, labels).item()
                val_correct += (logits.argmax(-1) == labels).sum().item(); val_total += labels.size(0)
            except Exception as e:
                print(f"  Skip val batch: {e}")

    val_acc = val_correct / val_total * 100
    history["val_acc"].append(val_acc)
    history["val_loss"].append(val_loss / len(val_loader))
    print(f"Epoch {epoch+1}: train={train_acc:.2f}%  val={val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), CKPT_DIR / f"best_mfft_{VARIANT}.pt")
        print(f"  ✅ Best: {best_acc:.2f}%")

    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
        ckpt_path = CKPT_DIR / f"checkpoint_epoch_{epoch+1}.pt"
        env.save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch, best_acc, history)
        env.upload_checkpoint(ckpt_path, f"{VARIANT}_model")

print(f"\nBest val accuracy: {best_acc:.2f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7: Upload to HuggingFace
# ═══════════════════════════════════════════════════════════
print("Uploading to HuggingFace...")
env.upload_checkpoint(CKPT_DIR / f"best_mfft_{VARIANT}.pt", f"{VARIANT}_model")

_manifest_path = PROJECT_ROOT / "dataset" / "metadata" / "train_manifest.csv"
if _manifest_path.exists(): env.upload_manifest(_manifest_path)

DEPLOY_DIR = PROJECT_ROOT / "paper" / "result" / f"{VARIANT}_model_deploy"
env.export_for_deployment(model, VARIANT, DEPLOY_DIR, best_acc=best_acc)
env.upload_deployment_model(DEPLOY_DIR, VARIANT)
print("HF upload complete")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8: Test Evaluation + Figures + Tables
# ═══════════════════════════════════════════════════════════
from src.visualize import generate_all_figures
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.calibration import calibration_curve
import csv, json

FIGS_DIR   = PROJECT_ROOT / "paper" / "result" / "full_scale" / f"{VARIANT}_model" / "fig"
TABLES_DIR = PROJECT_ROOT / "paper" / "result" / "full_scale" / f"{VARIANT}_model" / "table"
FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

all_test_labels, all_test_probs = [], []
model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test eval"):
        probs = F.softmax(model(images.to(device)), dim=-1)
        all_test_labels.extend(labels.cpu().numpy())
        all_test_probs.extend(probs[:, 1].cpu().numpy())

y_true  = np.array(all_test_labels)
y_score = np.array(all_test_probs)
y_pred  = (y_score >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)
all_labels = [s[1] for s in full_dataset.samples]

sample_img = None
for cls_dir in ["real", "BigGAN", "genimage_ai", "DALL-E3"]:
    d = PROJECT_ROOT / "dataset" / "images" / cls_dir
    if d.exists():
        files = list(d.rglob("*.jpg")) or list(d.rglob("*.png"))
        if files: sample_img = str(files[0]); break

generate_all_figures(history=history, train_loader=train_loader, model=model,
    val_loader=test_loader, device=device, y_true=y_true, y_score=y_score,
    cm=cm, labels=all_labels, sample_image_path=sample_img, output_dir=FIGS_DIR)

def _ser(x): return float(x) if isinstance(x, (np.floating, np.integer)) else (x.tolist() if isinstance(x, np.ndarray) else str(x))
def save_json(fn, data):
    with open(TABLES_DIR / fn, "w") as f: json.dump(data, f, indent=2, default=_ser)
    print(f"  {fn}")
def save_csv(fn, headers, rows):
    with open(TABLES_DIR / fn, "w", newline="") as f:
        w = csv.writer(f); w.writerow(headers); w.writerows(rows)
    print(f"  {fn}")

acc  = (y_pred == y_true).mean() * 100
prec = precision_score(y_true, y_pred, zero_division=0) * 100
rec  = recall_score(y_true, y_pred, zero_division=0) * 100
f1   = f1_score(y_true, y_pred, zero_division=0) * 100
auc  = roc_auc_score(y_true, y_score)
prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy="uniform")
ece  = float(np.mean(np.abs(prob_true - prob_pred)))
metrics = {"accuracy_pct":round(acc,2), "precision_pct":round(prec,2), "recall_pct":round(rec,2),
           "f1_score_pct":round(f1,2), "auc_roc":round(auc,4), "ece":round(ece,4)}

save_json("table3_evaluation_metrics.json", metrics)
save_csv("table3_evaluation_metrics.csv", ["Metric","Value"],
         [[k.replace("_"," ").title(), str(v)] for k,v in metrics.items()])
save_json("table2_training_history.json", history)

env.upload_figures_and_metrics(
    PROJECT_ROOT / "paper" / "result" / "full_scale" / f"{VARIANT}_model",
    f"{VARIANT}_model"
)

print(f"\n{'='*60}")
print(f"MFFT-{VARIANT.title()}: acc={acc:.2f}% | auc={auc:.4f} | best={best_acc:.2f}%")
print(f"{'='*60}")
print("\n✅ MFFT-Base complete. Proceed to 03_mfft_large.ipynb")